# BC Dataset Collection — Google Colab

Собирает один чанк (100k пар) от HeuristicAgent и сохраняет на Google Drive.

**Workflow:**
1. Запусти ячейки сверху вниз
2. Измени `PART_NAME` для каждой новой сессии (`part1`, `part2`, ...)
3. После завершения скачай файлы с Drive на локальную машину
4. Локально: `python training/combine_datasets.py --parts models/bc_part1.npz ... --output models/bc_dataset.npz`

In [ ]:
# ── Настройки ──────────────────────────────────────────────────────────────
PART_NAME  = "part1"       # Меняй для каждой новой сессии: part1, part2, ...
N_SAMPLES  = 100_000       # Пар на одну сессию
DRIVE_DIR  = "/content/drive/MyDrive/RL_practice/datasets"
# ───────────────────────────────────────────────────────────────────────────

In [ ]:
# Монтируем Google Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Клонируем репо
!git clone https://github.com/Andrew82mm/RL_practice.git /content/RL_practice
%cd /content/RL_practice

In [ ]:
# Устанавливаем зависимости
!pip install -q sb3-contrib gymnasium torch scipy tqdm cython numpy

In [ ]:
# Компилируем Cython-расширение для быстрого BFS
!python setup_fast.py build_ext --inplace

# Проверяем что скомпилилось
try:
    import block_puzzle_env._fast
    print("Cython _fast: OK — BFS будет быстрым")
except ImportError:
    print("Cython _fast: не загрузился — будет медленный Python BFS")

In [ ]:
# Проверяем что окружение импортируется
import sys
sys.path.insert(0, "/content/RL_practice")

from block_puzzle_env.environment import BlockPuzzleEnv
from evaluation.baselines import HeuristicAgent
print("Импорт OK")

env = BlockPuzzleEnv()
obs, _ = env.reset()
print(f"obs shape: {obs.shape}")

In [ ]:
# Создаём папку на Drive
import os
os.makedirs(DRIVE_DIR, exist_ok=True)

dataset_path = f"{DRIVE_DIR}/bc_{PART_NAME}.npz"
print(f"Данные будут сохранены: {dataset_path}")

In [ ]:
# Запускаем сбор данных
# Данные пишутся через memmap напрямую на Drive — RAM не расходуется
!python training/bc_train.py \
    --only-collect \
    --n-samples {N_SAMPLES} \
    --dataset-path "{dataset_path}"

In [ ]:
# Проверяем что файлы сохранились
obs_path = dataset_path.replace(".npz", "_obs.dat")
act_path = dataset_path.replace(".npz", "_act.dat")

for f in [dataset_path, obs_path, act_path]:
    size = os.path.getsize(f) / 1024**2
    print(f"{f}  →  {size:.1f} MB")

print("\nГотово! Скачай файлы с Google Drive на локальную машину.")